In [ ]:
import numpy as np
import sklearn
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import imageio.v3 as iio
import os
from PIL import Image
import imagehash


# Data loading and Processing

In [ ]:
Image_size = 96
Batch = 32
Random_seed = 42

In [ ]:
train_Tri_raw = tf.keras.utils.image_dataset_from_directory(
    "Training_Tri/",
    validation_split=0.30,
    subset="training",
    seed=Random_seed,
    image_size=(Image_size, Image_size),
    batch_size=None
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    "Training_Tri/",
    validation_split=0.30,
    subset="validation",
    seed=Random_seed,
    image_size=(Image_size, Image_size),
    batch_size=None
)


val_size = int(0.5 * len(temp_ds))
val_Tri_raw = temp_ds.take(val_size)
test_Tri_raw = temp_ds.skip(val_size)


In [ ]:
def focal_loss_sparse(gamma, alpha):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        # One-hot encode y_true
        y_true_one_hot = tf.one_hot(y_true, depth=y_pred.shape[-1])
        
        # Cross entropy
        ce = -tf.reduce_sum(y_true_one_hot * tf.math.log(y_pred), axis=-1)
        
        # Focal weight
        p_t = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        focal_weight = alpha * tf.pow(1 - p_t, gamma)
        
        return tf.reduce_mean(focal_weight * ce)
    return loss

def apply_blur(img):
    img = tf.expand_dims(img, 0)  
    img = tf.nn.avg_pool2d(img, ksize=3, strides=1, padding='SAME')
    return tf.squeeze(img, 0) 

def augment(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))    
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.6)
    image = tf.image.random_hue(image, max_delta=0.05)

    # Haze
    haze_intensity = tf.random.uniform((), 0.05, 0.25)
    haze = tf.ones_like(image) * 0.7
    image = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: image * (1 - haze_intensity) + haze * haze_intensity,
        lambda: image
    )

    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def compute_class_weights_Tri(ds):
    labels = np.concatenate([y.numpy() for _, y in ds.batch(512)])    
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    class_weights = dict(zip(classes, weights))
    print(f"Class weights: {class_weights}")
    return class_weights

In [ ]:
train_ds = (
    train_Tri_raw
    .cache() 
    .shuffle(5000, seed=Random_seed)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_Tri_raw
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .prefetch(tf.data.AUTOTUNE)
)
test_ds = (
    test_Tri_raw
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .prefetch(tf.data.AUTOTUNE)
)

class_weights = compute_class_weights_Tri(train_Tri_raw)


In [ ]:
model = keras.models.Sequential([
    
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
   
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.3),
   
    keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.GlobalAveragePooling2D(),
    
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(3, activation='softmax') 
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=focal_loss_sparse(gamma=2.0, alpha=0.25), # Use your custom loss
    metrics=['accuracy',
             keras.metrics.Recall(class_id=0, name='recall_fire'), # Changed 1 to 0
        keras.metrics.Precision(class_id=0, name='precision_fire')]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7
    ),
    keras.callbacks.ModelCheckpoint(
        'best_fire_model.keras',
        monitor='val_recall_fire',  
        save_best_only=True,
        mode='max'                 
    )
]

history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    class_weight= class_weights,
    callbacks=callbacks
)

In [ ]:
loss, acc, recall, precision = model.evaluate(test_ds)
print(f"Test Loss:      {loss:.4f}")
print(f"Test Accuracy:  {acc:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test Precision: {precision:.4f}")

y_pred_prob = model.predict(test_ds)

pred_classes = np.argmax(y_pred_prob, axis=1)

# Get true labels
true_labels = np.concatenate([y.numpy() for _, y in test_ds])

# Confusion matrix
cm = confusion_matrix(true_labels, pred_classes)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    true_labels,
    pred_classes,
    target_names=['Fire', 'Lakes ','No_Fire'] 
))